# 04 - Linear Regression (Baseline Model)

Train a linear regression baseline on `data/processed/traffic_density.csv` to predict `vehicle_count` for the next interval, using the engineered lag and rolling features.

Output: metrics appended to `results/model_comparison.csv`, model saved to `models/linear_regression.pkl`

**Split:** S01, S03, S04 = train · S02 = validation · S05 = test

In [1]:
import sys
import time
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression

sys.path.append("..")

from src.data_utils import (
    drop_incomplete_lag_rows,
    feature_columns,
    load_density_data,
    scale_features,
    split_by_column,
    target_column,
)
from src.eval_metrics import log_results, print_metrics, regression_metrics
from src.sequence_utils import create_sequences, flatten_for_baseline

In [2]:
processed_dir = Path("../data/processed")
input_path = processed_dir / "traffic_density.csv"

results_path = Path("../results/model_comparison.csv")
model_dir = Path("../models")
model_path = model_dir / "linear_regression.pkl"
window_size = 2

In [3]:
density = load_density_data(input_path)
density = drop_incomplete_lag_rows(density)

print(f"Loaded {len(density)} rows from {input_path}")
density.head()

Loaded 597 rows from ..\data\processed\traffic_density.csv


,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position,vehicle_count_smoothed,lag_1,lag_2,lag_3,lag_4,rate_of_change,peak_density,rolling_mean,rolling_std,density_category
0,S01,c001,train,4,3.333333,15,60,60,0.307692,2.777778,2.044444,1.644444,1.666667,2.200000,0.733333,2.777778,2.155556,0.574779,moderate
1,S01,c001,train,5,1.133333,15,75,75,0.384615,2.622222,2.777778,2.044444,1.644444,1.666667,-0.155556,2.777778,2.481481,0.386394,moderate
2,S01,c001,train,6,3.333333,15,90,90,0.461538,2.600000,2.622222,2.777778,2.044444,1.644444,-0.022222,2.777778,2.666667,0.096864,moderate
3,S01,c001,train,7,3.266667,15,105,105,0.538462,2.577778,2.600000,2.622222,2.777778,2.044444,-0.022222,2.622222,2.600000,0.022222,moderate
4,S01,c001,train,8,3.133333,15,120,120,0.615385,3.244444,2.577778,2.600000,2.622222,2.777778,0.666667,3.244444,2.807407,0.378648,moderate


In [4]:
train_df, val_df, test_df = split_by_column(density)
train_df, val_df, test_df, scaler = scale_features(train_df, val_df, test_df)

print(f"train: {len(train_df)} rows, val: {len(val_df)} rows, test: {len(test_df)} rows")

train: 132 rows, val: 41 rows, test: 424 rows


In [5]:
x_train_seq, y_train = create_sequences(train_df, window_size)
x_val_seq, y_val = create_sequences(val_df, window_size)
x_test_seq, y_test = create_sequences(test_df, window_size)

x_train = flatten_for_baseline(x_train_seq)
x_val = flatten_for_baseline(x_val_seq)
x_test = flatten_for_baseline(x_test_seq)

flattened_feature_names = [
    f"{col}_t-{window_size - 1 - step}"
    for step in range(window_size)
    for col in feature_columns
]

print(f"train sequences: {x_train_seq.shape}, val sequences: {x_val_seq.shape}, test sequences: {x_test_seq.shape}")
print(f"flattened feature vector length: {x_train.shape[1]}")

train sequences: (105, 2, 11), val sequences: (33, 2, 11), test sequences: (386, 2, 11)
flattened feature vector length: 22


In [6]:
model = LinearRegression()

start_time = time.time()
model.fit(x_train, y_train)
training_time_sec = time.time() - start_time

print(f"Trained in {training_time_sec:.4f} seconds")

Trained in 0.0013 seconds


In [7]:
val_preds = model.predict(x_val)
val_metrics = regression_metrics(y_val, val_preds)

print_metrics("linear_regression", "validation", val_metrics)
log_results(results_path, "linear_regression", "validation", val_metrics, training_time_sec)

linear_regression [validation] -> mae: 1.2952  rmse: 1.5967  r2: 0.3066  mape: 34.90%


In [8]:
test_preds = model.predict(x_test)
test_metrics = regression_metrics(y_test, test_preds)

print_metrics("linear_regression", "test", test_metrics)
log_results(results_path, "linear_regression", "test", test_metrics, training_time_sec)

linear_regression [test] -> mae: 1.7940  rmse: 2.1779  r2: 0.5933  mape: 99.85%


In [9]:
coefficients = pd.DataFrame({"feature": flattened_feature_names, "coefficient": model.coef_})
coefficients.sort_values("coefficient", key=abs, ascending=False)

,feature,coefficient
8,peak_density_t-1,8.554134
1,relative_position_t-1,7.593263
12,relative_position_t-0,-6.156399
19,peak_density_t-0,4.131612
10,rolling_std_t-1,-1.778779
9,rolling_mean_t-1,-1.274144
14,lag_1_t-0,-1.261760
2,vehicle_count_smoothed_t-1,-1.256869
15,lag_2_t-0,-1.246439
3,lag_1_t-1,-1.241271


In [10]:
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_path)

print(f"Saved model to {model_path}")

Saved model to ..\models\linear_regression.pkl


In [11]:
loaded_model = joblib.load(model_path)

sample_x = x_test[0:1]
sample_actual = y_test[0]
sample_pred = loaded_model.predict(sample_x)[0]

print(f"actual: {sample_actual:.4f}")
print(f"predicted: {sample_pred:.4f}")

actual: 4.8667
predicted: 4.8773


In [12]:
predictions_path = Path("../results/predictions.csv")

test_preds_final = model.predict(x_test)

predictions_df = pd.DataFrame({
    "model_name": "linear_regression",
    "split": "test",
    "actual": y_test,
    "predicted": test_preds_final,
})

if predictions_path.exists():
    existing_preds = pd.read_csv(predictions_path)
    existing_preds = existing_preds[existing_preds["model_name"] != "linear_regression"]
    predictions_df = pd.concat([existing_preds, predictions_df], ignore_index=True)

predictions_df.to_csv(predictions_path, index=False)
print(f"Saved test predictions to {predictions_path}")

Saved test predictions to ..\results\predictions.csv
